# 02 — E0 validation gate (run BEFORE spending GPU-days)
Administers IPIP-NEO-120 to every E1 persona configuration (both arms if the LoRA is served).
**Gate: mean Spearman r >= 0.60** (reference: r=.80-.90 for much larger models, Serapio-García et al. 2025).
Pre-registered rule: if P-arm fails and T-arm passes, T-arm becomes primary.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
print("project root:", ROOT)

project root: /home/raad/Papers/socialLLMs/traitmix


In [2]:
import numpy as np, pandas as pd, yaml, glob
from traitmix.data import load_ipip
from traitmix.llm import make_llm
from traitmix.questionnaire import administer, convergent_validity
from traitmix import personality as pers

SMOKE = False          # True = MockLLM + demo battery (pipeline test only)
llm = make_llm({"backend": "mock"} if SMOKE else
               {"backend": "vllm", "model": "meta-llama/Llama-3.1-8B-Instruct"})
items, battery = load_ipip(demo_ok=SMOKE)
print("battery:", battery, len(items), "items | backend:", llm.name)

FileNotFoundError: /home/raad/Papers/socialLLMs/traitmix/data/ipip/ipip_neo_120.csv not found. Download the public-domain IPIP-NEO-120 items from https://ipip.ori.org (or J.A. Johnson's IPIP-NEO materials), save as CSV with columns item_text,trait,keyed. Demo battery is only allowed for smoke tests (demo_ok=True).

In [ ]:
# target battery = the 11 E1 mu-vectors x 3 sampled agents each
cfgs = sorted(glob.glob(str(ROOT / "configs" / "e1" / "*.yaml")))
targets, measured, rows_long = [], [], []
rng = np.random.default_rng(0)
for f in cfgs:
    from traitmix.utils import load_config
    comp = load_config(f)["composition"]
    for rep in range(3):
        th = pers.sample_society(comp, 1, rng)[0]
        persona = {"name": f"Val_{len(targets)}", "age": 35, "occ": "analyst"}
        sc = administer(llm, th, persona, items, induction="prompt")
        targets.append(th); measured.append([sc[t] for t in pers.TRAITS])
        for k, t in enumerate(pers.TRAITS):
            rows_long.append({"trait": t, "target": th[k], "measured": sc[t]})
val = convergent_validity(np.array(targets), np.array(measured))
pd.DataFrame(rows_long).to_csv(ROOT / "results" / "e0_measured_vs_target.csv", index=False)
pd.DataFrame([{"arm": "prompt", **val}]).to_csv(ROOT / "results" / "e0_validation.csv", index=False)
val

In [ ]:
GATE = 0.60
print("PASS — proceed to full runs" if val["mean_r"] >= GATE else
      "FAIL — do NOT burn GPU-days: train/serve the T-arm (notebook 03), rerun with induction='tags';"
      " if both fail, fall back to binary high/low granularity and report honestly.")

Drift is measured **continuously in-run** (`trait_drift_mean` in every results row) and can be
re-checked post-hoc; optional mid-run questionnaire re-administration can be added to pilot runs.